# 03 — Model Training

This notebook trains and evaluates the classical ML models:
- **Logistic Regression** (simple baseline)
- **XGBoost** (strong ML baseline)

All models are trained on **all feature sets** directly in the loop — no separate showcase model.

Results are saved to `outputs/results/metrics.csv`.  
Plots are saved to `outputs/figures/models/`.

## 0 · Imports

In [1]:
import sys
sys.path.append('..')

from sklearn.model_selection import train_test_split

'''Self modules'''
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.models import split_data, get_X_y
from src.logistic_regression import train_logistic_regression
from src.xgboost import train_xgboost
from src.tuning import tune_logistic_regression, tune_xgboost
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve
from config import FEATURE_SETS, FEATURE_CONFIG

## 1 · Load & Split

Split is done on the **raw** DataFrame before preprocessing.  
This prevents any data leakage from imputation or encoding statistics.

In [2]:
df_raw = load_data()

df_train_raw, df_test_raw = split_data(df_raw, test_seasons=[2022, 2023])

Cache found - loading data from: C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\data\cache\pbp_raw.parquet
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py", line 2170, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\ultratb.py", line 1182, in structured_traceback
    return FormattedTB.structured_traceback(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\ultratb.py", line 1053, in structured_traceback
    return VerboseTB.structured_traceback(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rapha\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\ultratb.py", line 861, in structured_traceback
    formatted_exceptions: list[list[str]] = self.format_exception_as_a_whole(
                                        

## 2 · Feature Set Loop

Preprocess the largest feature set once, then slice subsets from it to avoid redundant work.

Both models are trained on **all feature sets** and results are collected.
Results are appended to `metrics.csv` — existing rows for the same
`(model, feature_set)` combination are automatically overwritten.

In [ ]:
# preprocess the largest feature set once; reuse column subsets to avoid redundant work.
LARGEST_FS = max(FEATURE_SETS, key=lambda k: len(FEATURE_SETS[k]))
_df_train_full = preprocess(df_train_raw, feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)
_df_test_full  = preprocess(df_test_raw,  feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)

# align test on train columns (missing = 0, extra = dropped)
_df_test_full = _df_test_full.reindex(columns=_df_train_full.columns, fill_value=0)

In [ ]:
# update feature sets with encoded column names before the loop
FEATURE_SETS_ENCODED = {
    fs_name: [
        c for c in _df_train_full.columns
        if c != "target" and any(c == f or c.startswith(f + "_") for f in fs_cols)
    ]
    for fs_name, fs_cols in FEATURE_SETS.items()
}
FEATURE_SETS_ENCODED

In [ ]:
# ================================================================
# MAIN LOOP  –  feste Hyperparameter (Baseline)
# ================================================================
for fs_name, fs_cols in FEATURE_SETS_ENCODED.items():

    print(f"\n{'='*55}")
    print(f" Feature set: {fs_name}  ({len(fs_cols)} features)")
    print(f"{'='*55}")

    _df_train = _df_train_full[fs_cols + ["target"]]
    _df_test  = _df_test_full[fs_cols  + ["target"]]
    _X_train, _y_train = get_X_y(_df_train)
    _X_test,  _y_test  = get_X_y(_df_test)

    # Logistic Regression
    _lr  = train_logistic_regression(_X_train, _y_train)
    evaluate_model(_lr, _X_test, _y_test, "LogisticRegression", fs_name)
    plot_confusion_matrix(_lr, _X_test, _y_test, "LogisticRegression", fs_name)

    # XGBoost
    _xgb = train_xgboost(_X_train, _y_train)
    evaluate_model(_xgb, _X_test, _y_test, "XGBoost", fs_name)
    plot_confusion_matrix(_xgb, _X_test, _y_test, "XGBoost", fs_name)

    plot_roc_curve(
        models={"LogisticRegression": _lr, "XGBoost": _xgb},
        X_test=_X_test,
        y_test=_y_test,
        feature_set=fs_name,
    )

print("\nDone. Baseline complete.")

In [ ]:
# ================================================================
# TUNING LOOP  –  optional, läuft nach dem Baseline Loop
# ================================================================
for fs_name, fs_cols in FEATURE_SETS_ENCODED.items():

    print(f"\n{'='*55}")
    print(f" TUNING  |  Feature set: {fs_name}  ({len(fs_cols)} features)")
    print(f"{'='*55}")

    _df_train = _df_train_full[fs_cols + ["target"]]
    _df_test  = _df_test_full[fs_cols  + ["target"]]
    _X_train, _y_train = get_X_y(_df_train)
    _X_test,  _y_test  = get_X_y(_df_test)

    # Logistic Regression
    _lr_params    = tune_logistic_regression(_X_train, _y_train)
    _lr_tuned     = train_logistic_regression(_X_train, _y_train, **_lr_params)
    evaluate_model(_lr_tuned, _X_test, _y_test, "LR_tuned", fs_name)
    plot_confusion_matrix(_lr_tuned, _X_test, _y_test, "LR_tuned", fs_name)

    # XGBoost
    _xgb_params   = tune_xgboost(_X_train, _y_train)
    _xgb_tuned    = train_xgboost(_X_train, _y_train, **_xgb_params)
    evaluate_model(_xgb_tuned, _X_test, _y_test, "XGBoost_tuned", fs_name)
    plot_confusion_matrix(_xgb_tuned, _X_test, _y_test, "XGBoost_tuned", fs_name)

    plot_roc_curve(
        models={"LR_tuned": _lr_tuned, "XGBoost_tuned": _xgb_tuned},
        X_test=_X_test,
        y_test=_y_test,
        feature_set=fs_name,
    )

print("\nDone. Tuning complete.")